# Livrable modélisation


---
## Identification du problème

### Contexte

Nous sommes une entreprise de grande distribution avec des magasins un peu partout en France. Nous avons un entrepôt centralisé d'où partent nos camions de livraison pour approvisionner nos différents magasins. L'objectif est donc que tous les magasins reçoivent les marchandises dans les temps. Pour cela, nous avons besoin d'optimiser les tournées de nos camions pour réduire les coûts de transport, limiter les émissions de CO2 et améliorer l'efficacité de notre logistique.

Ce projet de recherche opérationnelle s’inscrit dans un contexte de transition écologique visant à réduire les émissions et optimiser les transports. Nous faisons partie de l’équipe CesiCDP chargée de répondre à un appel de l’ADEME sur la mobilité intelligente. Notre mission consiste à modéliser et résoudre un problème d’optimisation de tournées de livraison afin de limiter les déplacements et la consommation des véhicules.

---

## Définition mathématique du problème

### Définition Formelle du problème

Le problème mTSP peut-être modélisé à l'aide de la théorie des graphes. Nous considérons un graphe non orienté de réseau routier $G = (V, A)$.

#### Les ensembles

- $V=\{0,1,...,n\}$ : L'ensemble des sommets où 0 est le dépôt (point de départ)
- $A=\{(i,j):i,j \in V,i \neq j\}$ : L'ensemble des arcs représentant les routes possibles entre les villes.

#### Les paramètres

- $c_{ij}$ : le coût (la durée ou la distance) associé au passage sur l'arc $(i,j)$
    - Gestion de la restriction de passage : Si une route est interdite (travaux, sens interdit), on pose $c_{ij} = M \text{ Où } M \gg \sum c_{ij}$. Si elle est simplement plus coûteuse, on ajuste la valeur de $c_{ij}$ en conséquence.
- $m$ : Le nombre de véhicules utilisés pour couvrir l'ensemble des villes.
#### Les Variables de Décision
- $x_{ij}$ : Variable binaire égale à 1 si un véhicule emprunte l'arc $(i, j)$, 0 sinon.
- $u_i$ : Variable auxiliaire pour l'élimination des sous-tours (représentant l'ordre de passage).

---

### Formulation Mathématique

L'objectif est de trouver un ensemble de $m$ tournées minimisant le coût global, tout en s'assurant que chaque ville (hors dépôt) est visitée exactement une fois.

#### La fonction objectif

La fonction objectif vise à minimiser la somme totale des coûts (ou durées) de tous les déplacements effectués par la flotte de véhicules :

$$\text{Minimiser } Z = \sum_{i \in V} \sum_{j \in V, j \neq i} c_{ij} x_{ij}$$

#### Les contraintes

1. Visite unique des clients : Chaque ville $j$ (autre que le dépôt) doit être visitée et quittée exactement une fois :

$$\sum_{i \in V, i \neq j} x_{ij} = 1 \quad \forall j \in V'$$

$$\sum_{j \in V, j \neq i} x_{ij} = 1 \quad \forall i \in V'$$

2. Flux au dépôt : Exactement $m$ véhicules doivent quitter le dépôt et $m$ véhicules doivent y revenir :

$$\sum_{j \in V'} x_{0j} = m$$

$$\sum_{i \in V'} x_{i0} = m$$

3. Élimination des sous-tours (Contraintes MTZ) : Pour éviter la formation de boucles fermées qui ne passent pas par le dépôt, on utilise les contraintes de Miller-Tucker-Zemlin :

$$u_i - u_j + (n-m+1)x_{ij} \leq n-m \quad \forall (i, j) \in A, i, j \neq 0 \\
\text{Où } 1 \leq u_i \leq n \text{ pour tout } i \in V' \text{.}$$

#### Le programme linéaire

$$PL=\begin{cases}
\text{Minimiser } Z = \sum_{i \in V} \sum_{j \in V, j \neq i} c_{ij} x_{ij} \\
\text{s.c.} \\
\sum_{i \in V, i \neq j} x_{ij} = 1 \quad \forall j \in V' \\
\sum_{j \in V, j \neq i} x_{ij} = 1 \quad \forall i \in V' \\
\sum_{j \in V'} x_{0j} = m \\
\sum_{i \in V'} x_{0j} = m \\
u_i - u_j + (n-m+1)x_{ij} \leq n-m \quad \forall (i, j) \in A, i, j \neq 0 \\
\text{Où } 1 \leq u_i \leq n \text{ pour tout } i \in V' \text{.}
\end{cases}$$

---

## Représentation des données

Pour les données, nous avons pris un jeu de donnée sur simplemap.com ([lien](https://simplemaps.com/data/fr-cities)). Les données contiennent des villes ainsi que leur coordonnée géographique (latitude et longitude) ce qui va nous permettre de pouvoir calculer la distance entre deux villes grâce à la formule [d'haversine](https://fr.wikipedia.org/wiki/Formule_de_haversine).

Puisque nous devons générer les données de manière aléatoire, nous avons décidé d'utiliser des données au départ réelles puisque nous utiliserons les données depuis la dataset précédent, mais en sélectionnant les villes de manières aléatoires. En fonctionnant de cette manière, nous avons à la fois des données réelles avec des vraies villes, mais choisis de façon aléatoire.

Dans ce contexte, l'utilisation de graphe est la plus logique pour représenter les données et les traitées. Nous avons donc avec ces données, décidé de représenter :
- Les sommets du graphes par les différentes villes.
- Les arêtes par le trajet entre deux sommets, villes.
- Le poids des arêtes représente le nombre de kilomètres entre deux sommets/villes.

---
## Complexité

### Étude théorique de la complexité : Rigueur mathématique

Avant de commencer à coder notre algorithme pour l'ADEME, il est essentiel de prouver mathématiquement à quel point notre problème est difficile. En informatique théorique, cette étape est cruciale : si l'on prouve que le problème est "trop complexe" pour être résolu de manière parfaite, on justifie scientifiquement le fait d'utiliser des algorithmes d'approximation par la suite.

Pour mener cette démonstration nous allons procéder en quatre étapes.

#### 1. Transformer notre besoin en "Problème de décision"

**Pourquoi faire cela ?** En mathématiques, il est très difficile de manipuler le concept d'optimisation absolue ("trouver le plus court chemin"). La méthode académique standard consiste à transformer notre recherche d'optimum en une question fermée (réponse par OUI ou NON) par rapport à un seuil donné. 

**Formalisation mathématique :**
Définissons une *instance* (un cas de test) de notre problème :
- Un réseau sous forme de graphe $G = (V, E)$, où $V$ est l'ensemble des villes et $E$ les routes praticables (les routes bloquées n'existent pas dans $E$).
- Une flotte de $K$ véhicules de livraison, avec $K \in \mathbb{N}$.
- Une fonction de coût $c : E \rightarrow \mathbb{R}^+$.
- Un budget maximum autorisé $B \in \mathbb{R}^+$ (en kilomètres ou en temps).

**La question de décision :**  
*Existe-t-il un ensemble de $K$ tournées partant du dépôt, visitant chaque client de $V$ exactement une fois via les routes $E$, de telle sorte que le coût total soit inférieur ou égal au budget $B$ ?*

Ce qui s’écrit :

$$
\sum_{i=1}^{K} \sum_{(u,v) \in T_i} c(u,v) \leq B
$$

---

#### 2. Prouver que notre problème appartient à la "Classe NP"

**Qu'est-ce que cela signifie ?** Un problème appartient à la classe NP (Non-déterministe Polynomial) si, lorsqu'on nous donne une solution "devinée" (un *certificat*), nous pouvons vérifier très rapidement si elle est correcte ou non.

**Démonstration :** Supposons que l'on nous fournisse une solution candidate (la liste des villes visitées).  

On vérifie :
1. Chaque ville est visitée exactement une fois :
$$
\forall v \in V, \quad \text{visité une seule fois}
$$

2. Chaque déplacement respecte le graphe :
$$
(u,v) \in E
$$

3. Le coût total respecte le budget :
$$
C(S) = \sum c(u,v) \leq B
$$

Cette vérification nécessite de parcourir les données une seule fois.Le temps de calcul est donc directement proportionnel au nombre de villes n. En mathématiques, on dit que la vérification se fait en temps polynomial O(n). 
$$
T(n) = O(n)
$$

**Conclusion de l'étape 2 :**  
Puisque la vérification est rapide :

$$
\text{Problème} \in NP
$$

---
- définition formelle NP → [2] Chapitre 2
- reformulation pédagogique → [1] Chapitre 34
- notion de vérification polynomial → [2] p.31–67
- complexité algorithmique → [1]
- Basé sur définition NP → [1][2]
#### 3. Démonstration de la NP-Complétude (La Preuve par Réduction)

**L'objectif :** prouver que notre problème fait partie des plus difficiles de NP via une réduction polynomiale. Nous allons réduire le problème du Cycle Hamiltonien (trouver un chemin passant par tous les points d'un graphe, reconnu NP-complet) vers le problème du Voyageur de Commerce (TSP), qui est la base de notre projet.

Nous utilisons :

$$
\text{Cycle Hamiltonien} \leq_p \text{TSP}
$$

**A. La transformation :**

- Si la route existait dans le graphe d'origine G, on lui donne une distance (un poids) de 1.
On construit un graphe complet $G'$ :
$$
E' = V \times V
$$
-  Si la route a été inventée, on lui donne un poids de 2 (pour la pénaliser) :
$$
c(u,v) =
\begin{cases}
1 & \text{si } (u,v) \in E \\
2 & \text{sinon}
\end{cases}
$$

- On fixe notre budget maximum B pour qu'il soit exactement égal au nombre de villes |V|:
$$
B = |V|
$$

**B. La preuve par double implication :**

- *Sens 1 :* S'il existe un vrai Cycle Hamiltonien dans le graphe d'origine G, le camion pourra faire le tour en n'empruntant que les routes d'origine, donc uniquement des routes de poids 1. Son coût total sera donc exactement |V| $\times 1 = B$. La réponse au problème est OUI. 
Le coût est :

$$
C = |V| \times 1 = |V|
$$

- *Sens 2 :* S'il existe une tournée de coût $\le B$, alors le camion a l'interdiction mathématique d'utiliser une route inventée de poids 2. S'il en utilisait ne serait-ce qu'une seule, le coût minimal monterait à 2 + (|V|-1) = |V| + 1, ce qui dépasse le budget. Il n'a donc emprunté que des routes de poids 1 (les routes d'origine). Cela prouve qu'un Cycle Hamiltonien existe. 
Si une arête de coût 2 est utilisée :

$$
2 + (|V|-1) = |V| + 1 > B
$$

Donc impossible → seules les arêtes d'origine sont utilisées.

- réductions polynomiales → [2] Chapitre 3
- preuve TSP NP-complet → [1] Chapitre 34
- construction classique de réduction → [1][2]
- preuve formelle → [1] Chapitre 34
- classification NP-complet → [2]
- VRP = généralisation du TSP → [2] Chapitre 8

**Conclusion :**

$$
\text{TSP} \in NP\text{-complet}
$$


**D. Adaptation à notre cas d'étude :**
Notre problème pour l'ADEME est beaucoup plus complexe que le TSP de base : nous avons plusieurs camions (K) et un graphe incomplet (routes interdites). Puisque notre modèle englobe le TSP comme un sous-cas extrêmement simplifié (le cas où K=1 et toutes les routes sont ouvertes), il est logiquement au moins aussi difficile.

Mathématiquement parlant : 
Notre problème généralise le TSP :

$$
\text{TSP} \subseteq \text{VRP}
$$

Donc :

$$
\text{VRP} \geq_p \text{TSP}
\Rightarrow \text{VRP (décision)} \in NP\text{-complet}
$$

---

#### 4. Conclusion finale : Le problème d'Optimisation est NP-Difficile

Pour l'ADEME, notre but final n'est pas de répondre "OUI" ou "NON" à un budget, mais de trouver mathématiquement la tournée la plus courte possible (optimisation absolue). Chercher l'optimum d'un problème NP-complet nous fait basculer dans la classe des problèmes NP-Difficiles.
Notre objectif réel est :

$$
\min C(S)
$$

Relation clé :

$$
\text{TSP décision} \leq_p \text{TSP optimisation}
$$

Donc :

$$
\text{Optimisation} \in NP\text{-difficile}
$$

Si nous utilisons la force brute, le nombre de solutions est :

$$
(n-1)!
$$

Donc :

$$
T(n) = O(n!)
$$

Pour un réseau réel, cela devient intractable (explosion combinatoire).

**Quelles sont les conséquences concrètes pour notre code Python ?**

Le postulat fondamental de l'informatique théorique (P \neq NP) implique qu'il n'existe à ce jour aucun algorithme au monde capable de trouver la solution parfaite à notre problème en un temps de calcul raisonnable (polynomial). Si nous essayions d'utiliser une méthode exacte (comme la force brute, qui évalue toutes les combinaisons de routes), l'espace des solutions augmenterait de manière factorielle O(n!). Pour un réseau comme celui de la France, l'ordinateur mettrait des milliards d'années à terminer son calcul (c'est ce qu'on appelle l'explosion combinatoire).

Cette barrière mathématique indépassable justifie la stratégie que nous adopterons dans les prochaines boucles du projet : nous n'utiliserons pas de méthodes de résolution exactes. Nous allons coder des heuristiques et méta-heuristiques. Ces algorithmes font le compromis d'abandonner la quête de la solution "parfaite à 100%" au profit d'excellentes solutions très proches de l'optimum, calculées en seulement quelques secondes, ce qui répondra parfaitement au besoin opérationnel de l'ADEME.$$

---

**Conclusion :**

$$
P \neq NP \Rightarrow \text{pas d'algorithme polynomial exact connu}
$$

Nous utiliserons donc :

$$
\text{Heuristiques} + \text{Méta-heuristiques}
$$

---
## Contraintes

### Choix des contraintes à ajouter au projet
Les contraintes que nous avons choisis de rajouter au projets sont:
- Coût ou restriction de passage sur certaines arêtes : Certaines routes peuvent être plus coûteuses ou interdites (par exemple, travaux ou routes bloquées).
- Utilisation de plusieurs véhicules : Il peut y avoir plusieurs sous-tournées plutôt qu'une seule grande.

### Explication de ces contraintes

 Dans le problème classique du Voyageur de Commerce, l'objectif algorithmique est de trouver un unique cycle hamiltonien optimal. C'est-à-dire une seule et unique boucle fermée qui visite chaque sommet du graphe exactement une fois, avant de revenir à son point de départ. En introduisant une flotte de K véhicules, la nature même de la solution recherchée dans le graphe change :
 - Génération de multiples cycles : La solution n'est plus un cycle hamiltonien unique, mais un ensemble de N cycles distincts.
  - Le sommet central : Tous ces cycles partagent obligatoirement un et un seul sommet en commun : le nœud d'origine (le dépôt). C'est le point d'intersection de toutes les tournées.
- Sommets disjoints : À l'exception du sommet d'origine, tous les autres sommets du graphe doivent être répartis entre ces les différents cycles. Un sommet appartenant au cycle du véhicule A ne peut pas appartenir au cycle du véhicule B. L'algorithme doit donc non seulement optimiser l'ordre de passage, mais aussi faire un choix de partitionnement des sommets.


Dans un réseau routier réel, des aléas surviennent quotidiennement (travaux, routes barrées, embouteillages importants). Dans notre modélisation, ces aléas ne modifient pas les sommets (les villes restent à leur place), mais viennent directement altérer les arêtes du graphe de deux manières concrètes :
 - Les restrictions totales : Concrètement, cela se traduit par la suppression de X arêtes dans notre graphe. Les algorithmes de résolution ne pourront tout simplement plus emprunter ces chemins. Nous allons supprimer de manières aléatoires des arêtes tout en vérifiant de garder un graphe connexe.
- Les pénalités de coût : Dans ce cas, l'arête existe toujours, mais son "poids" (qui représente le temps de trajet ou le coût) est artificiellement gonflé. Nous allons cibler X  arêtes dont nous allons multiplier le poids par un coefficient de pénalité. Le graphe reste le même, mais les "plus courts chemins" varieront, forçant l'algorithme à réévaluer si un long détour géométrique n'est pas finalement devenu "plus court" en temps qu'une ligne droite embouteillée.

## Références

[1] Thomas H. Cormen et al., *Introduction to Algorithms*, MIT Press, 2009, Chapitre 34, pp. 1025–1085.
https://www.cs.mcgill.ca/~akroit/math/compsci/Cormen%20Introduction%20to%20Algorithms.pdf

[2] Michael R. Garey, David S. Johnson, *Computers and Intractability*, W.H. Freeman, 1979, Chapitres 2, 3, 8.
https://perso.limos.fr/~palafour/PAPERS/PDF/Garey-Johnson79.pdf

---
Réalisé par :
FISA A3 CESI 2025-2026
- Hugo  Denombret
- Thomas Courtot
- Arthur Roux
- Yulia Pavlova
- Evann Abrial